# Cell2location Analysis: Human Glioblastoma
**Goal:** Map GBM cell states (MES-like, NPC-like, OPC-like, AC-like) and tumor microenvironment cell types to spatial locations

**Reference:** GBmap - Harmonized atlas of 39,355 GBM cells from 110 patients (Ruiz-Moreno et al. 2022)

**Spatial data:** Human glioblastoma Visium dataset (10x Genomics, 3,468 spots)

## 0. GPU Setup
Must run this cell first, before importing theano.

In [ ]:
import os
os.environ["THEANO_FLAGS"] = "device=cuda,floatX=float32"

import theano
print(f"Theano device: {theano.config.device}")

import scanpy as sc
import pandas as pd
import numpy as np
import torch

print(f"PyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Imports and Paths

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import cell2location
from cell2location.models import RegressionGeneBackgroundCoverageTorch, LocationModelLinearDependentW

# Set up directories
DATA_DIR = "/projectnb/ds596/projects/Team_7/data/humanGlioblastoma"
REF_DIR = "/projectnb/ds596/projects/Team_7/data/GBmap_reference"
RESULTS_DIR = "/projectnb/ds596/projects/Team_7/results/human_gbm"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Reference directory: {REF_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## 2. Load GBmap Reference Data
GBmap provides curated cell type annotations for 14 GBM cell populations.

In [ ]:
COMPRESSED_REF = os.path.join(REF_DIR, "GBmap_reference_compressed.h5ad")

if os.path.exists(COMPRESSED_REF):
    print("Loading pre-compressed reference...")
    adata_ref = sc.read(COMPRESSED_REF)
else:
    print("Creating compressed reference from original files (one-time setup)...")
    
    # Load metadata
    metadata = pd.read_csv(
        os.path.join(REF_DIR, "GBmap_metadata.csv.gz"), 
        compression='gzip'
    )
    
    # Load count matrix
    counts = pd.read_csv(
        os.path.join(REF_DIR, "GBmap_raw_counts.tsv.gz"), 
        compression='gzip', sep='\t', index_col=0
    )
    
    # Create AnnData (cells x genes)
    adata_ref = ad.AnnData(
        X=counts.T.values.astype(np.int32),
        obs=metadata,
        var=pd.DataFrame(index=counts.index)
    )
    
    # Add cell type labels
    adata_ref.obs['cell_type'] = adata_ref.obs['predicted.high_hierarchy']
    
    # Filter low-quality cells and genes
    sc.pp.filter_cells(adata_ref, min_genes=200)
    sc.pp.filter_genes(adata_ref, min_cells=3)
    
    # Save for faster loading next time
    adata_ref.write(COMPRESSED_REF)

print(f"Reference shape: {adata_ref.shape[0]:,} cells, {adata_ref.shape[1]:,} genes")
print(f"Cell types: {adata_ref.obs['cell_type'].nunique()}")
print("\nCell type distribution:")
for ct, count in adata_ref.obs['cell_type'].value_counts().head(10).items():
    print(f"  {ct}: {count:,} cells")

# Verify we have key GBM cell types
gbm_types = ['MES-like', 'NPC-like', 'OPC-like', 'AC-like', 'TAM-MG', 'TAM-BDM']
present = [ct for ct in gbm_types if ct in adata_ref.obs['cell_type'].values]
print(f"\nGBM cell types present: {present}")

## 3. Load Visium Spatial Data

In [ ]:
adata_spatial = sc.read_visium(
    DATA_DIR,
    count_file='Parent_Visium_Human_Glioblastoma_filtered_feature_bc_matrix.h5',
    load_images=True
)

print(f"Spatial data: {adata_spatial.shape[0]:,} spots, {adata_spatial.shape[1]:,} genes")
print(f"Tissue image available: {list(adata_spatial.uns['spatial'].keys())}")

## 4. Align Reference and Spatial Data
Find common genes between the two datasets and subset to those.

In [ ]:
# Fix duplicate gene names
adata_ref.var_names_make_unique()
adata_spatial.var_names_make_unique()

# Find overlapping genes
common_genes = np.intersect1d(adata_spatial.var_names, adata_ref.var_names)
print(f"Common genes: {len(common_genes):,}")
print(f"  {len(common_genes)/adata_spatial.n_vars*100:.1f}% of spatial genes")
print(f"  {len(common_genes)/adata_ref.n_vars*100:.1f}% of reference genes")

# Subset to common genes
adata_ref = adata_ref[:, common_genes].copy()
adata_spatial = adata_spatial[:, common_genes].copy()

# Save raw counts for later
adata_spatial.layers["counts"] = adata_spatial.X.copy()

print(f"\nAligned reference: {adata_ref.shape}")
print(f"Aligned spatial: {adata_spatial.shape}")

## 5. Create Reference Signatures
Calculate average gene expression for each cell type. This gives the model a "profile" for each cell population.

In [ ]:
# Convert cell type to categorical for grouping
adata_ref.obs['cell_type'] = adata_ref.obs['cell_type'].astype('category')
cell_types = adata_ref.obs['cell_type'].cat.categories.tolist()
print(f"Cell types ({len(cell_types)}): {cell_types}")

# Normalize for averaging (log transform)
sc.pp.normalize_total(adata_ref, target_sum=1e4)
sc.pp.log1p(adata_ref)

# Compute mean expression per cell type
print("\nCalculating mean expression per cell type...")
mean_expression = []
for ct in cell_types:
    cells_in_type = adata_ref[adata_ref.obs['cell_type'] == ct]
    mean_exp = cells_in_type.X.mean(axis=0)
    if hasattr(mean_exp, 'A1'):
        mean_exp = mean_exp.A1
    mean_expression.append(mean_exp)
    print(f"  {ct}: {cells_in_type.n_obs} cells")

# Build signature matrix (genes × cell types)
ref_signatures = pd.DataFrame(
    np.array(mean_expression).T,
    index=adata_ref.var_names,
    columns=cell_types
)

print(f"\nReference signature matrix: {ref_signatures.shape[0]:,} genes × {ref_signatures.shape[1]} cell types")
ref_signatures.to_csv(os.path.join(RESULTS_DIR, "reference_signatures.csv"))

## 6. Train Regression Model
This learns refined cell type signatures from the reference data.

In [ ]:
# Check GPU status
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU available")

# Add dummy covariate (required by the model)
adata_ref.obs['dummy_covar'] = '1'

# Prepare input data
cell2covar = pd.DataFrame({
    "patient": adata_ref.obs["patient"].values,
    "cell_type": adata_ref.obs["cell_type"].values,
    "dummy_covar": adata_ref.obs["dummy_covar"].values
}, index=adata_ref.obs_names)

X_data_ref = adata_ref.X.copy()
if hasattr(X_data_ref, "toarray"):
    X_data_ref = X_data_ref.toarray()
X_data_ref = X_data_ref.astype(np.float32)

# Set training parameters (adjust as needed)
N_ITER = 100  # Increase for better results

print("Building regression model...")
mod_ref = RegressionGeneBackgroundCoverageTorch(
    sample_id="patient",
    cell2covar=cell2covar,
    X_data=X_data_ref,
    data_type='float32',
    n_iter=N_ITER,
    learning_rate=0.005,
    total_grad_norm_constraint=200,
    verbose=True,
    use_cuda=True,
    minibatch_size=2500,
    use_average_as_initial_value=True
)

print(f"Model built: {mod_ref.n_obs} cells, {mod_ref.n_var} genes, {mod_ref.n_fact} factors")

# Train
print("Training regression model...")
mod_ref.fit_advi_iterative(n=1, n_type='restart', n_iter=N_ITER, train_proportion=0.9)

# Sample posterior
print("Sampling posterior...")
mod_ref.sample_posterior(node='all', n_samples=500, save_samples=False, mean_field_slot='init_1')

# Extract cell type signatures
inf_aver_array = mod_ref.samples['post_sample_means']['gene_factors']
fact_names = mod_ref.fact_names

# Keep only cell type factors (skip patient and dummy factors)
cell_type_factors = [f for f in fact_names if f.startswith('cell_type_')]
factor_indices = [i for i, f in enumerate(fact_names) if f.startswith('cell_type_')]
inf_aver_filtered = inf_aver_array[factor_indices, :]
cell_type_names = [f.replace('cell_type_', '') for f in cell_type_factors]

inf_aver = pd.DataFrame(
    inf_aver_filtered.T,
    index=adata_ref.var_names,
    columns=cell_type_names
)

print(f"\nCell type signatures: {inf_aver.shape}")
print(f"Cell types: {inf_aver.columns.tolist()}")

# Verify we have the expected GBM types
present = [ct for ct in gbm_types if ct in inf_aver.columns]
print(f"\nGBM cell types extracted: {present}")

## 7. Spatial Deconvolution
Map cell types to spatial locations using the learned signatures.

In [ ]:
# Prepare spatial data
X_data_spatial = adata_spatial.X.copy()
if hasattr(X_data_spatial, "toarray"):
    X_data_spatial = X_data_spatial.toarray()
X_data_spatial = X_data_spatial.astype(np.float32)

# Align signatures with spatial genes
common_genes_spatial = np.intersect1d(adata_spatial.var_names, inf_aver.index)
inf_aver_aligned = inf_aver.loc[common_genes_spatial, :].copy()
adata_spatial_aligned = adata_spatial[:, common_genes_spatial].copy()

print(f"Aligned spatial data: {adata_spatial_aligned.shape}")
print(f"Aligned signatures: {inf_aver_aligned.shape}")

cell_state_mat = inf_aver_aligned.values.astype(np.float32)

# Set parameters (adjust for speed vs quality)
N_ITER_SPATIAL = 10  # Increase for better results
N_COMB = 5

print("\nBuilding spatial model...")
mod_spatial = LocationModelLinearDependentW(
    cell_state_mat=cell_state_mat,
    X_data=X_data_spatial,
    n_comb=N_COMB,
    data_type='float32',
    n_iter=N_ITER_SPATIAL,
    learning_rate=0.005,
    total_grad_norm_constraint=200,
    verbose=True,
    cell_number_prior={'cells_per_spot': 15, 'factors_per_spot': 10, 'combs_per_spot': 3}
)

print(f"Spatial model built: {mod_spatial.n_obs} spots, {mod_spatial.n_var} genes, {mod_spatial.n_fact} cell types")

# Train spatial model
print("Training spatial model...")
mod_spatial.fit_advi_iterative()

# Sample posterior for uncertainty estimates
print("Sampling posterior...")
mod_spatial.sample_posterior(
    node='all',
    n_samples=20,
    save_samples=False,
    mean_field_slot='init_1',
    batch_size=50
)

# Extract cell type abundances per spot
spot_factors = mod_spatial.samples['post_sample_means']['spot_factors']
print(f"Spot factors shape: {spot_factors.shape} (spots × cell types)")

# Add abundances to AnnData object
cell_type_names = inf_aver_aligned.columns.tolist()
for i, ct in enumerate(cell_type_names):
    adata_spatial.obs[f'{ct}_mean'] = spot_factors[:, i]

# Identify dominant cell type per spot
abundance_cols = [f'{ct}_mean' for ct in cell_type_names]
dominant_idx = np.argmax(adata_spatial.obs[abundance_cols].values, axis=1)
adata_spatial.obs['dominant_cell_type'] = [cell_type_names[i] for i in dominant_idx]

# Print summary
print("\n" + "="*60)
print("SPATIAL MAPPING RESULTS")
print("="*60)
print(f"Total spots: {adata_spatial.n_obs:,}")
print(f"Cell types mapped: {len(cell_type_names)}")
print(f"\nDominant cell type distribution (top 10):")
for ct, count in adata_spatial.obs['dominant_cell_type'].value_counts().head(10).items():
    pct = count / adata_spatial.n_obs * 100
    print(f"  {ct}: {count:,} spots ({pct:.1f}%)")

## 8. Save Results

In [ ]:
# Save full AnnData
out_path = os.path.join(RESULTS_DIR, "adata_human_gbm_cell2location.h5ad")
adata_spatial.write_h5ad(out_path)
print(f"Saved: {out_path}")

# Save summary table
summary_df = pd.DataFrame({
    'Cell Type': cell_type_names,
    'Total Abundance': [adata_spatial.obs[f'{ct}_mean'].sum() for ct in cell_type_names],
    'Mean per Spot': [adata_spatial.obs[f'{ct}_mean'].mean() for ct in cell_type_names],
    'Std per Spot': [adata_spatial.obs[f'{ct}_mean'].std() for ct in cell_type_names],
    '% Spots Positive': [(adata_spatial.obs[f'{ct}_mean'] > 0).mean() * 100 for ct in cell_type_names]
})
summary_df = summary_df.sort_values('Total Abundance', ascending=False)
summary_df.to_csv(os.path.join(RESULTS_DIR, "cell_abundance_summary.csv"), index=False)
print(f"Saved summary: {os.path.join(RESULTS_DIR, 'cell_abundance_summary.csv')}")

## 9. Visualization: Cell Type Composition

In [ ]:
total_abundance = adata_spatial.obs[abundance_cols].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
colors = plt.cm.viridis(np.linspace(0, 1, len(total_abundance)))
bars = ax.bar(range(len(total_abundance)), total_abundance.values, color=colors)

ax.set_xticks(range(len(total_abundance)))
ax.set_xticklabels(total_abundance.index, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Estimated Total Cell Abundance')
ax.set_title('Cell Type Composition in Human Glioblastoma')
ax.set_yscale('log')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "cell_type_composition.png"), dpi=300, bbox_inches='tight')
plt.show()

## 10. Visualization: Spatial Maps of Key Cell Types

In [ ]:
# Select key cell types for visualization
gbm_malignant = ['MES-like', 'NPC-like', 'OPC-like', 'AC-like']
tme_types = ['TAM-MG', 'TAM-BDM', 'Endothelial', 'CD4/CD8']

visualize_types = [ct for ct in gbm_malignant + tme_types if f'{ct}_mean' in adata_spatial.obs.columns]
print(f"Visualizing {len(visualize_types)} cell types: {visualize_types}")

# Create multi-panel spatial plot
n_cols = 4
n_rows = (len(visualize_types) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for idx, ct in enumerate(visualize_types):
    sc.pl.spatial(
        adata_spatial,
        color=f'{ct}_mean',
        cmap='magma',
        ax=axes[idx],
        title=ct,
        size=1.2,
        show=False
    )
    axes[idx].set_title(ct, fontsize=12)

# Turn off unused axes
for idx in range(len(visualize_types), len(axes)):
    axes[idx].axis('off')

plt.suptitle('Spatial Distribution of GBM Cell States and TME', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "spatial_maps_gbm_states.png"), dpi=300, bbox_inches='tight')
plt.show()

## 11. Visualization: Dominant Cell Type Map

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

sc.pl.spatial(
    adata_spatial,
    color='dominant_cell_type',
    ax=ax,
    title='Dominant Cell Type per Spatial Spot',
    size=1.2,
    legend_loc='right margin',
    show=False
)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dominant_cell_type_map.png"), dpi=300, bbox_inches='tight')
plt.show()

## 12. Regional Heterogeneity Analysis
Group spots by their cell type composition to identify distinct tumor regions.

In [ ]:
from sklearn.cluster import KMeans

# Cluster spots based on cell type composition
abundance_matrix = adata_spatial.obs[abundance_cols].values

# Find clusters
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
adata_spatial.obs['region_cluster'] = kmeans.fit_predict(abundance_matrix)

# Plot spatial clusters
fig, ax = plt.subplots(1, 2, figsize=(20, 10))

sc.pl.spatial(
    adata_spatial,
    color='region_cluster',
    cmap='Set1',
    ax=ax[0],
    title='Spatial Regions (K-means)',
    size=1.0,
    show=False
)

# Heatmap of cell type composition by region
region_composition = pd.DataFrame()
for region in sorted(adata_spatial.obs['region_cluster'].unique()):
    mask = adata_spatial.obs['region_cluster'] == region
    comp = adata_spatial.obs[abundance_cols][mask].mean()
    region_composition[f'Region_{region}'] = comp

top_types = total_abundance.head(15).index
sns.heatmap(region_composition.loc[top_types], cmap='RdYlBu_r', 
            annot=True, fmt='.2f', ax=ax[1],
            xticklabels=True, yticklabels=True)
ax[1].set_title('Cell Type Composition by Spatial Region')
ax[1].set_ylabel('Cell Type')

plt.suptitle('Regional Heterogeneity in Human Glioblastoma', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "regional_heterogeneity.png"), dpi=300, bbox_inches='tight')
plt.show()

## 13. Cell Type Correlation Analysis

In [ ]:
corr_matrix = adata_spatial.obs[abundance_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=True, fmt='.2f', 
            square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Cell Type Abundance Correlations')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "cell_type_correlations.png"), dpi=300, bbox_inches='tight')
plt.show()

## 14. Summary Statistics

In [ ]:
print(f"\nDataset Summary:")
print(f"  Total spots analyzed: {adata_spatial.n_obs:,}")
print(f"  Genes used: {adata_spatial.n_vars:,}")
print(f"  Cell types mapped: {len(cell_type_names)}")

print(f"\nTop 10 Most Abundant Cell Types:")
for ct, val in total_abundance.head(10).items():
    pct = val / total_abundance.sum() * 100
    print(f"  {ct:20s}: {val:,.0f} total ({pct:.1f}%)")

print(f"\nOutput Files:")
print(f"  Results directory: {RESULTS_DIR}")
for f in os.listdir(RESULTS_DIR):
    if f.endswith('.png') or f.endswith('.csv') or f.endswith('.h5ad'):
        size = os.path.getsize(os.path.join(RESULTS_DIR, f)) / 1024
        print(f"  {f}: {size:.1f} KB")

# Parameter Sweep Analysis
We tested five parameter configurations to see how they affect results. Runs are ordered from fastest to slowest.
## Parameter Configurations

In [ ]:
configs = {
    # Fastest, for quick testing
    'run1_test': {
        'name': 'Test Run (Baseline)',
        'regression': {'n_iter': 100, 'learning_rate': 0.005},
        'spatial': {'n_iter': 500, 'n_comb': 20, 'learning_rate': 0.005},
        'posterior': {'n_samples': 200, 'batch_size': 200}
    },
    # More posterior samples for better uncertainty estimates
    'run2_more_samples': {
        'name': 'More Posterior Samples',
        'regression': {'n_iter': 100, 'learning_rate': 0.005},
        'spatial': {'n_iter': 500, 'n_comb': 20, 'learning_rate': 0.005},
        'posterior': {'n_samples': 500, 'batch_size': 500}
    },
    # Higher learning rate to test convergence speed
    'run3_higher_lr': {
        'name': 'Higher Learning Rate',
        'regression': {'n_iter': 100, 'learning_rate': 0.025},
        'spatial': {'n_iter': 500, 'n_comb': 20, 'learning_rate': 0.025},
        'posterior': {'n_samples': 200, 'batch_size': 200}
    },
    # Intermediate settings - good balance of speed and quality
    'run4_intermediate': {
        'name': 'Intermediate Run',
        'regression': {'n_iter': 500, 'learning_rate': 0.005},
        'spatial': {'n_iter': 5000, 'n_comb': 30, 'learning_rate': 0.005},
        'posterior': {'n_samples': 500, 'batch_size': 500}
    },
    # Highest quality, for final results
    'run5_final': {
        'name': 'Final Run (High Quality)',
        'regression': {'n_iter': 2000, 'learning_rate': 0.005},
        'spatial': {'n_iter': 10000, 'n_comb': 30, 'learning_rate': 0.005},
        'posterior': {'n_samples': 500, 'batch_size': 500}
    }
}

# Create results directory
RUNS_DIR = os.path.join(RESULTS_DIR, "parameter_runs")
os.makedirs(RUNS_DIR, exist_ok=True)

print("Parameter configurations:")
for key, config in configs.items():
    print(f"\n{key}: {config['name']}")
    print(f"  Regression: {config['regression']['n_iter']} iters, LR={config['regression']['learning_rate']}")
    print(f"  Spatial: {config['spatial']['n_iter']} iters, n_comb={config['spatial']['n_comb']}")
    print(f"  Posterior: {config['posterior']['n_samples']} samples")

## Function to Run Cell2location with Given Parameters

In [ ]:
def run_cell2location_gpu(config_name, config, adata_ref, adata_spatial):
    """Run Cell2location with specified parameters using GPU when available."""
    
    print("="*70)
    print(f"RUNNING: {config['name']}")
    print("="*70)
    
    # Check GPU
    use_cuda = torch.cuda.is_available()
    if use_cuda:
        print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    
    run_dir = os.path.join(RUNS_DIR, config_name)
    os.makedirs(run_dir, exist_ok=True)
    
    # ----- Regression Model -----
    print("\n--- Training Regression Model ---")
    print(f"  Iterations: {config['regression']['n_iter']}")
    print(f"  Learning rate: {config['regression']['learning_rate']}")
    
    adata_ref.obs['dummy_covar'] = '1'
    
    cell2covar = pd.DataFrame({
        "patient": adata_ref.obs["patient"].values,
        "cell_type": adata_ref.obs["cell_type"].values,
        "dummy_covar": adata_ref.obs["dummy_covar"].values
    }, index=adata_ref.obs_names)
    
    X_data_ref = adata_ref.X.copy()
    if hasattr(X_data_ref, "toarray"):
        X_data_ref = X_data_ref.toarray()
    X_data_ref = X_data_ref.astype(np.float32)
    
    mod_ref = RegressionGeneBackgroundCoverageTorch(
        sample_id="patient",
        cell2covar=cell2covar,
        X_data=X_data_ref,
        data_type='float32',
        n_iter=config['regression']['n_iter'],
        learning_rate=config['regression']['learning_rate'],
        total_grad_norm_constraint=200,
        verbose=True,
        use_cuda=use_cuda,
        minibatch_size=2500,
        use_average_as_initial_value=True
    )
    
    mod_ref.fit_advi_iterative(n=1, n_type='restart', 
                               n_iter=config['regression']['n_iter'], 
                               train_proportion=0.9)
    mod_ref.sample_posterior(node='all', n_samples=500, save_samples=False, mean_field_slot='init_1')
    
    # Extract cell type signatures
    inf_aver_array = mod_ref.samples['post_sample_means']['gene_factors']
    fact_names = mod_ref.fact_names
    cell_type_factors = [f for f in fact_names if f.startswith('cell_type_')]
    factor_indices = [i for i, f in enumerate(fact_names) if f.startswith('cell_type_')]
    inf_aver_filtered = inf_aver_array[factor_indices, :]
    cell_type_names = [f.replace('cell_type_', '') for f in cell_type_factors]
    
    inf_aver = pd.DataFrame(
        inf_aver_filtered.T,
        index=adata_ref.var_names,
        columns=cell_type_names
    )
    
    # ----- Spatial Mapping -----
    print("\n--- Training Spatial Model ---")
    print(f"  Iterations: {config['spatial']['n_iter']}")
    print(f"  n_comb: {config['spatial']['n_comb']}")
    print(f"  Learning rate: {config['spatial']['learning_rate']}")
    
    X_data_spatial = adata_spatial.X.copy()
    if hasattr(X_data_spatial, "toarray"):
        X_data_spatial = X_data_spatial.toarray()
    X_data_spatial = X_data_spatial.astype(np.float32)
    
    common_genes_spatial = np.intersect1d(adata_spatial.var_names, inf_aver.index)
    inf_aver_aligned = inf_aver.loc[common_genes_spatial, :].copy()
    cell_state_mat = inf_aver_aligned.values.astype(np.float32)
    
    mod_spatial = LocationModelLinearDependentW(
        cell_state_mat=cell_state_mat,
        X_data=X_data_spatial,
        n_comb=config['spatial']['n_comb'],
        data_type='float32',
        n_iter=config['spatial']['n_iter'],
        learning_rate=config['spatial']['learning_rate'],
        total_grad_norm_constraint=200,
        verbose=True,
        cell_number_prior={'cells_per_spot': 15, 'factors_per_spot': 10, 'combs_per_spot': 3}
    )
    
    mod_spatial.fit_advi_iterative()
    mod_spatial.sample_posterior(
        node='all',
        n_samples=config['posterior']['n_samples'],
        save_samples=False,
        mean_field_slot='init_1',
        batch_size=config['posterior']['batch_size']
    )
    
    spot_factors = mod_spatial.samples['post_sample_means']['spot_factors']
    
    # Save results
    adata_results = adata_spatial.copy()
    cell_type_names = inf_aver_aligned.columns.tolist()
    for i, ct in enumerate(cell_type_names):
        adata_results.obs[f'{ct}_mean'] = spot_factors[:, i]
    
    output_path = os.path.join(run_dir, "results.h5ad")
    adata_results.write_h5ad(output_path)
    
    # Summary
    abundance_cols = [f'{ct}_mean' for ct in cell_type_names]
    total_abundance = adata_results.obs[abundance_cols].sum().sort_values(ascending=False)
    
    summary_df = pd.DataFrame({
        'Cell Type': total_abundance.index,
        'Total Abundance': total_abundance.values,
        'Percentage': total_abundance.values / total_abundance.sum() * 100
    })
    summary_df.to_csv(os.path.join(run_dir, "summary.csv"), index=False)
    
    print(f"\n✓ Results saved to: {run_dir}")
    print(f"  Top 5 cell types: {summary_df.head(5)['Cell Type'].tolist()}")
    
    return adata_results, summary_df

## Run Selected Configurations
Choose which configurations to run by uncommenting them below.

In [ ]:
# Select which configs to run (uncomment as needed)
configs_to_run = [
    'run1_test',           # Fastest (~25 min)
    'run2_more_samples',   # More posterior samples (~30 min)
    'run3_higher_lr',      # Higher learning rate (~25 min)
    'run4_intermediate',   # Good balance (~3.5 hours)
    'run5_final',          # Highest quality (~17 hours)
]
# Calculate estimated time
time_estimates = {
    'run1_test': 25,
    'run2_more_samples': 30,
    'run3_higher_lr': 25,
    'run4_intermediate': 210,
    'run5_final': 1020,
}

total_time = sum([time_estimates.get(c, 0) for c in configs_to_run])

print("PARAMETER SWEEP")
print(f"Configs to run ({len(configs_to_run)} total):")
for c in configs_to_run:
    print(f"  - {c}: {configs[c]['name']} (~{time_estimates.get(c, '?')} min)")
print(f"\nEstimated total time: {total_time} minutes ({total_time/60:.1f} hours)")
print(f"GPU available: {torch.cuda.is_available()}")
print("="*80)

confirm = input("\nType 'yes' to start: ")
if confirm.lower() != 'yes':
    print("Aborted.")
else:
    all_results = {}
    for idx, config_name in enumerate(configs_to_run, 1):
        print(f"\n--- Running {config_name} ({idx}/{len(configs_to_run)}) ---")
        results, summary = run_cell2location_gpu(config_name, configs[config_name], adata_ref, adata_spatial)
        all_results[config_name] = results

## Compare Results Across Configurations

In [ ]:
# Load summaries from completed runs
summaries = {}
for config_name in configs.keys():
    summary_path = os.path.join(RUNS_DIR, config_name, "summary.csv")
    if os.path.exists(summary_path):
        summaries[config_name] = pd.read_csv(summary_path)
        print(f"Loaded {config_name}: {len(summaries[config_name])} cell types")

# Create comparison plot
if summaries:
    fig, axes = plt.subplots(len(summaries), 1, figsize=(12, 4 * len(summaries)))
    if len(summaries) == 1:
        axes = [axes]
    
    for idx, (config_name, summary) in enumerate(summaries.items()):
        ax = axes[idx]
        top10 = summary.head(10)
        ax.bar(range(len(top10)), top10['Percentage'].values)
        ax.set_xticks(range(len(top10)))
        ax.set_xticklabels(top10['Cell Type'], rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Percentage (%)')
        ax.set_title(f"{configs[config_name]['name']}: Cell Type Composition")
        ax.set_yscale('log')
    
    plt.tight_layout()
    plt.savefig(os.path.join(RUNS_DIR, "comparison.png"), dpi=150, bbox_inches='tight')
    plt.show()

## Function to Generate Plots for a Run
Creates multiple visualization types including a 2x3 grid of key GBM cell types.

In [ ]:
def generate_plots_for_run(run_dir, run_name, cmap='winter'):
    """Generate summary plots for a completed run."""
    
    adata = sc.read_h5ad(os.path.join(run_dir, "results.h5ad"))
    
    # Get cell types
    cell_type_cols = [col for col in adata.obs.columns if col.endswith('_mean')]
    cell_types = [col.replace('_mean', '') for col in cell_type_cols]
    
    plots_dir = os.path.join(run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)
    
    # Figure 1: Composition bar plot
    total_abundance = adata.obs[cell_type_cols].sum().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(total_abundance)))
    bars = ax.bar(range(len(total_abundance)), total_abundance.values, color=colors)
    ax.set_xticks(range(len(total_abundance)))
    ax.set_xticklabels(total_abundance.index, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Estimated Total Cell Abundance')
    ax.set_title(f'{run_name}: Cell Type Composition')
    ax.set_yscale('log')
    
    for bar, val in zip(bars, total_abundance.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                f'{val:.0f}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "composition.png"), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Composition plot saved")
    
    # Figure 2: Dominant cell type map
    dominant_idx = np.argmax(adata.obs[cell_type_cols].values, axis=1)
    adata.obs['dominant_cell_type'] = [cell_types[i] for i in dominant_idx]
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sc.pl.spatial(
        adata,
        color='dominant_cell_type',
        ax=ax,
        title=f'{run_name}: Dominant Cell Type per Spot',
        size=1.0,
        legend_loc='right margin',
        show=False
    )
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "dominant_cell_type.png"), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Dominant cell type map saved")
    
    # Figure 3: All cell types grid
    n_cell_types = len(cell_type_cols)
    print(f"Creating figure with all {n_cell_types} cell types...")
    
    n_cols = 4
    n_rows = 4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
    axes = axes.flatten()
    
    sorted_cols = total_abundance.sort_values(ascending=False).index.tolist()
    
    for idx, ct_col in enumerate(sorted_cols):
        ct_name = ct_col.replace('_mean', '')
        sc.pl.spatial(
            adata,
            color=ct_col,
            cmap=cmap,
            ax=axes[idx],
            title=ct_name,
            size=0.8,
            show=False
        )
        axes[idx].set_title(ct_name, fontsize=10, fontweight='bold')
        pct = total_abundance[ct_col] / total_abundance.sum() * 100
        axes[idx].text(0.02, 0.98, f'{pct:.1f}% of total', 
                       transform=axes[idx].transAxes, fontsize=7,
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    for idx in range(len(sorted_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f'{run_name}: Spatial Distribution of All {n_cell_types} Cell Types', 
                 fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "all_cell_types_grid.png"), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"All cell types grid saved")
    
    # Figure 4: Six key cell types (2x3 grid)
    highlight_cell_types = ['MES-like', 'Astrocyte', 'CD4/CD8', 'OPC', 'AC-like', 'Pericyte']
    available_types = [ct for ct in highlight_cell_types if f'{ct}_mean' in adata.obs.columns]
    
    print(f" Creating 2x3 figure for {len(available_types)} key cell types...")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, ct in enumerate(available_types):
        ct_col = f'{ct}_mean'
        pct = total_abundance[ct_col] / total_abundance.sum() * 100
        
        sc.pl.spatial(
            adata,
            color=ct_col,
            cmap=cmap,
            ax=axes[idx],
            title=ct,
            size=0.8,
            show=False
        )
        axes[idx].set_title(ct, fontsize=12, fontweight='bold')
        axes[idx].text(0.02, 0.98, f'{pct:.1f}% of total', 
                       transform=axes[idx].transAxes, fontsize=8,
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    for idx in range(len(available_types), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f'{run_name}: Key GBM Cell Types', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "six_key_cell_types.png"), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Six key cell types figure saved")
    
    # Figure 5: Summary statistics table
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.axis('tight')
    ax.axis('off')
    
    summary_table = []
    for ct_col in sorted_cols:
        ct_name = ct_col.replace('_mean', '')
        pct = total_abundance[ct_col] / total_abundance.sum() * 100
        summary_table.append([
            ct_name,
            f"{total_abundance[ct_col]:.0f}",
            f"{pct:.1f}%",
            f"{adata.obs[ct_col].mean():.3f}",
            f"{(adata.obs[ct_col] > 0).mean() * 100:.1f}%"
        ])
    
    table = ax.table(cellText=summary_table,
                     colLabels=['Cell Type', 'Total Abundance', '% of Total', 'Mean/Spot', '% Spots Positive'],
                     loc='center',
                     cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    
    ax.set_title(f'{run_name}: Cell Type Summary Statistics', fontsize=14, fontweight='bold', y=0.95)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "summary_table.png"), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Summary table saved")
    
    print(f"\n All plots saved to: {plots_dir}")
    return adata

## Generate Plots for All Completed Runs

In [ ]:
# List of completed runs (update based on what you've run)
completed_runs = ['run1_test', 'run2_more_samples', 'run3_higher_lr', 'run4_intermediate', 'run5_final']

# Display names for plots
run_display_names = {
    'run1_test': 'Test Run',
    'run2_more_samples': 'More Posterior Samples',
    'run3_higher_lr': 'Higher Learning Rate',
    'run4_intermediate': 'Intermediate Run',
    'run5_final': 'Final Run'
}

# Colormap for spatial plots (cool tones, no purple (hard to see as H&E image is purple))
COLORMAP = 'winter'  # Options: 'winter', 'Blues', 'BuGn'

for config_name in completed_runs:
    run_dir = os.path.join(RUNS_DIR, config_name)
    results_file = os.path.join(run_dir, "results.h5ad")
    
    if os.path.exists(results_file):
        print(f"\n✓ Loading {config_name}")
        adata_loaded = sc.read_h5ad(results_file)
        generate_plots_for_run(
            run_dir,
            run_display_names.get(config_name, config_name),
            cmap=COLORMAP
        )
    else:
        print(f"\n✗ {config_name} not found at {results_file}")